# Lecture 5: Automatic differentiation — the Week 1 project

This last session of Week 1 is a **project**: you will build a small library computing
**exact derivatives of Python code**. It uses everything seen this week — classes (L1),
typing (L2), tests (L3), SOLID and TDD (L4) — and adds two Python tools: **operator
overloading** and **`@dataclass`**.

It is also the bridge to Week 2 (optimization), where every model is fitted by following
a gradient downhill — and where your library will check every gradient we write.

## Objectives
By the end of this session you can:
- explain four ways of computing a derivative and their limits (by hand, finite differences, symbolic, automatic);
- overload Python operators (`__add__`, `__radd__`, …, returning `NotImplemented` when needed) and use `@dataclass(frozen=True)`;
- explain **forward mode** with dual numbers `a + bε`, `ε² = 0`, and implement it;
- explain **reverse mode** (backpropagation): computation graph, adjoints, accumulation, topological order — and implement it;
- choose between forward and reverse mode, and explain why machine learning uses reverse mode.

## 1. Why derivatives?

Near a point `a`, a smooth function is almost a straight line:

$$f(a + h) \approx f(a) + f'(a)\,h .$$

The derivative tells you **which way is downhill** and how steep it is. For a function
of several variables `f: ℝⁿ → ℝ`, the vector of partial derivatives is the **gradient**
`∇f`, and `−∇f` is the direction of steepest descent. Training a model — from linear
regression to a neural network — means minimizing a loss, and that is done by following
gradients (Week 2).

So we need gradients that are **correct** and **cheap**. How do we get them?

## 2. Four ways to compute a derivative

| Method | Idea | Problem |
|---|---|---|
| **By hand** | derive the formula, code it | error-prone, and must be redone at each change of the model |
| **Finite differences** | `(f(x+h) − f(x))/h` | approximate; `n + 1` evaluations for a gradient |
| **Symbolic** | a computer-algebra system derives the expression (e.g. `sympy`) | expressions blow up; does not handle loops and `if`s |
| **Automatic differentiation** | apply the chain rule to each elementary operation, *while the code runs* | none of the above: exact to machine precision, cost of a few evaluations |

Finite differences deserve a closer look, because they are the reference we test against.

In [ ]:
import math

def f(x):
    return x * math.exp(x)

exact = 2 * math.e                     # f'(x) = (1 + x)eˣ, so f'(1) = 2e
for h in (1e-1, 1e-4, 1e-8, 1e-12, 1e-16):
    approx = (f(1 + h) - f(1)) / h
    print(f"h = {h:7.0e}   error = {abs(approx - exact):.2e}")

The error first **decreases** (the Taylor truncation error is `O(h)`), then **increases
again**: when `h` is tiny, `f(1 + h)` and `f(1)` are almost equal floating-point numbers,
and subtracting them loses all the significant digits (*catastrophic cancellation*). At
`h = 1e-16`, `1 + h == 1` in double precision and the "derivative" is `0`!

The best compromise is around `h ≈ √ε ≈ 1e-8` (`ε ≈ 2.2e-16` is the machine epsilon),
giving only about **8 correct digits**. Automatic differentiation gives all 16, with no `h`.

## 3. Two Python tools for today

### 3.1 Operator overloading

When Python evaluates `a + b`, it calls `a.__add__(b)`. You can define these **special
methods** ("dunder" methods) in your own classes:

| Expression | Method called | Reflected (if the left operand says `NotImplemented`) |
|---|---|---|
| `a + b` | `a.__add__(b)` | `b.__radd__(a)` |
| `a - b` | `a.__sub__(b)` | `b.__rsub__(a)` |
| `a * b` | `a.__mul__(b)` | `b.__rmul__(a)` |
| `a / b` | `a.__truediv__(b)` | `b.__rtruediv__(a)` |
| `a ** b` | `a.__pow__(b)` | `b.__rpow__(a)` |
| `-a` | `a.__neg__()` | — |

The protocol for mixed types: when `__add__` does not know how to handle `other`, it
**returns** (not raises!) the special value `NotImplemented`. Python then tries the
reflected method of the other operand, and raises `TypeError` only if both give up.
That is why `2 + x` needs `x.__radd__`: `int.__add__` knows nothing about your class.

In [ ]:
class Vec2:
    def __init__(self, x: float, y: float) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        return f"Vec2({self.x}, {self.y})"

    def __add__(self, other: "Vec2") -> "Vec2":
        if not isinstance(other, Vec2):
            return NotImplemented
        return Vec2(self.x + other.x, self.y + other.y)

    def __mul__(self, k: float) -> "Vec2":          # vector * scalar
        if not isinstance(k, (int, float)):
            return NotImplemented
        return Vec2(k * self.x, k * self.y)

    def __rmul__(self, k: float) -> "Vec2":         # scalar * vector
        return self * k


u, v = Vec2(1, 2), Vec2(3, 4)
print(u + v, u * 2, 2 * u)
try:
    u + 1
except TypeError as error:
    print("TypeError:", error)

### 3.2 `@dataclass`

Classes that mainly hold data need boilerplate: `__init__`, `__repr__`, `__eq__`… The
`@dataclass` **decorator** (module `dataclasses`) generates them from annotated fields.
With `frozen=True`, instances are **immutable**: any assignment raises
`FrozenInstanceError` — perfect for values like numbers, which should never change
behind your back.

In [ ]:
from dataclasses import dataclass, FrozenInstanceError

@dataclass(frozen=True)
class Point:
    x: float
    y: float = 0.0            # a default value, as for a function parameter

p = Point(1.0)
print(p, p == Point(1.0, 0.0))          # __repr__ and __eq__ for free
try:
    p.x = 3.0
except FrozenInstanceError as error:
    print("FrozenInstanceError:", error)

## 4. Forward mode: dual numbers

A **dual number** is `a + bε`, where `ε` is a new symbol with **`ε² = 0`** (but `ε ≠ 0`).
Arithmetic follows the usual rules, dropping every `ε²`:

- `(a + bε) + (c + dε) = (a + c) + (b + d)ε`
- `(a + bε)(c + dε) = ac + (ad + bc)ε + bd·ε² = ac + (ad + bc)ε`

Look at the `ε` part of the product: `ad + bc` is exactly the **product rule**
`(uv)' = u'v + uv'`. This is no coincidence. By Taylor's formula, since `ε² = 0`,

$$f(a + b\varepsilon) = f(a) + f'(a)\,b\,\varepsilon \quad\text{(exactly — all higher terms vanish).}$$

So if we evaluate a function at `x + 1·ε`, the result is `f(x) + f'(x)ε`: **the value and
the derivative travel together**. Each elementary operation just has to apply its rule:

| Operation | Value part | `ε` part |
|---|---|---|
| `u + v` | `u + v` | `u' + v'` |
| `u · v` | `u·v` | `u'v + uv'` |
| `u / v` | `u / v` | `(u'v − uv') / v²` |
| `uᵏ` (constant `k`) | `uᵏ` | `k·uᵏ⁻¹·u'` |
| `exp(u)` | `eᵘ` | `eᵘ·u'` |
| `log(u)` | `log u` | `u'/u` |
| `sin(u)` | `sin u` | `cos(u)·u'` |

The last three are the **chain rule** `(f∘u)' = f'(u)·u'`.

**By hand**: `f(x) = x·eˣ` at `x = 1`. Start from `1 + 1ε`. Then `exp` gives `e + eε`,
and the product gives `(1 + ε)(e + eε) = e + (e + e)ε = e + 2eε`. So `f'(1) = 2e` ✓.

Here is a *minimal* dual number with only `+` and `*` (you will write the complete one):

In [ ]:
class MiniDual:
    def __init__(self, val: float, der: float) -> None:
        self.val, self.der = val, der

    def __add__(self, other: "MiniDual") -> "MiniDual":
        return MiniDual(self.val + other.val, self.der + other.der)

    def __mul__(self, other: "MiniDual") -> "MiniDual":
        return MiniDual(self.val * other.val, self.der * other.val + self.val * other.der)


def g(x):
    three = MiniDual(3, 0)                 # a constant: derivative 0
    return x * x + three * x               # g(x) = x² + 3x, g'(x) = 2x + 3

result = g(MiniDual(2, 1))                 # evaluate at 2 + 1ε
print("g(2) =", result.val, "  g'(2) =", result.der)

Notice that **`g` is ordinary Python code**: it could contain loops, `if`s, function
calls. Automatic differentiation differentiates the **program actually executed**, not a
formula. (A subtle consequence: with an `if`, you get the derivative of the branch taken.)

**Cost.** One evaluation at `x + ε` gives **one** derivative. For `f: ℝⁿ → ℝ`, getting the
gradient requires `n` evaluations, seeding `ε` on one variable at a time. Fine for `n = 3`,
hopeless for a neural network with `n = 10⁶` parameters...

## 5. Reverse mode: backpropagation

Reverse mode records the computation as a **graph**, then applies the chain rule
**backwards**, from the output to the inputs.

Take `L = x·y + x` at `(x, y) = (2, 3)`. Name the intermediate results:

```
x = 2        y = 3
a = x · y    = 6        local derivatives:  ∂a/∂x = y = 3,   ∂a/∂y = x = 2
L = a + x    = 8        local derivatives:  ∂L/∂a = 1,       ∂L/∂x = 1
```

**Forward pass**: compute the values, and store for each node its parents with the
*local* derivative. **Backward pass**: compute the **adjoint** `v̄ = ∂L/∂v` of every node,
starting from `L̄ = 1`, with the chain rule

$$\bar{\text{parent}} \mathrel{+}= \frac{\partial\,\text{node}}{\partial\,\text{parent}} \times \bar{\text{node}} .$$

| Step | Node visited | Contributions |
|---|---|---|
| 0 | `L` | `L̄ = 1` |
| 1 | `L = a + x` | `ā += 1·1 = 1`,  `x̄ += 1·1 = 1` |
| 2 | `a = x·y` | `x̄ += 3·1 → x̄ = 4`,  `ȳ += 2·1 = 2` |

Result: `∂L/∂x = 4 = y + 1` and `∂L/∂y = 2 = x` ✓.

Two key points:
- **Accumulation (`+=`)**: `x` is used twice (in `a` and in `L`), so it receives **two**
  contributions. Forgetting `+=` is the classic bug.
- **Order**: a node may be processed only once **all** the nodes that use it have been
  processed. We visit the nodes in **reverse topological order** (a topological order
  lists every node after its parents).

**Cost.** One backward pass gives the derivative of `L` with respect to **all** inputs at
once, for a cost of a few evaluations — whatever `n`. The price is **memory**: the whole
graph is stored during the forward pass.

## 6. Forward or reverse?

| | Forward mode | Reverse mode |
|---|---|---|
| One pass gives | the derivative along **one input direction** | the gradient of **one output** |
| Gradient of `f: ℝⁿ → ℝ` | `n` passes | **1** pass |
| Derivatives of `f: ℝ → ℝᵐ` | **1** pass | `m` passes |
| Memory | small | stores the graph |
| Implementation | very simple | a bit more subtle |

In machine learning, a loss maps **millions of parameters to one number**: reverse mode
wins by far. That is **backpropagation**, the engine of PyTorch, TensorFlow and JAX.

## 7. Bonus: three algebras, and the complex step

Dual numbers are one of three "number systems" `a + b·t`, depending on `t² = s`:

| | `s = −1` | `s = 0` | `s = +1` |
|---|---|---|---|
| Name | complex (`i`) | **dual** (`ε`) | hyperbolic (`j`) |
| Defining form `Q(a + bt) = a² + s·b²`, signature | `(1, 1, 0)` | **`(1, 0, 1)`** | `(2, 0, 0)` |
| Norm `N = a² − s·b²` (multiplicative) | `a² + b²` | `a²` | `a² − b²` |
| Geometry of `N = 1` | circle (rotations) | two lines (shears) | hyperbola (boosts) |
| Structure | a field | nilpotent `ε` | zero divisors `(1+j)(1−j) = 0` |

Only `s = 0` truncates Taylor's series after the first order, which makes the derivative
**exact**. With `s = −1`, `Im f(x + ih)/h = f'(x) − h²f'''(x)/6 + …`: the **complex-step**
method, an `O(h²)` approximation *without subtraction*, hence without cancellation — so
`h` can be absurdly small:

In [ ]:
import cmath

def f_complex(z):
    return z * cmath.exp(z)

for h in (1e-4, 1e-8, 1e-20, 1e-200):
    approx = f_complex(1 + 1j * h).imag / h
    print(f"h = {h:7.0e}   error = {abs(approx - 2 * math.e):.2e}")

## 8. The project

Clone the repository `autodiff-lab` (see its `README.md`):

```
autodiff/
├── dual.py        Part 1 — forward mode (Dual)                 45 min
├── var.py         Part 2 — reverse mode (Var)                  95 min
├── gradient.py    Part 3 — gradients of f: ℝⁿ → ℝ              20 min
└── functions.py   exp, log, sin, cos for float / Dual / Var    (provided)
tests/             the specification, as unit tests             (provided)
notebooks/         plots, once your tests are green
```

- Work **test by test** (TDD, Lecture 4): `make part1`, then `make part2`, then `make part3`.
- In each class, `__add__` is written as a worked example: follow the same pattern.
- `make check` runs all the tests **and** `mypy --strict`. Both must be green.
- Commit after each group of tests that turns green.
- Then run `notebooks/autodiff-plots.ipynb` and answer its questions.

Design notes (SOLID):
- `Dual` and `Var` are independent classes, each with a single responsibility (SRP).
- `functions.py` works on `float`, `Dual` and `Var` alike by delegating to their methods
  (duck typing, Lecture 2): adding a new number type does not modify it (OCP).
- Each operation stores its own local derivatives: the `backward` engine never needs to
  know which operation created a node. A new operation is a new method, nothing else (OCP).

## What's next

Next week we stop *computing* gradients and start *following* them: fitting linear and
logistic regressions, then stochastic training. Every hand-written gradient of Week 2
will be checked against the library you build today.